# 🤖 Agentic AI — Module 1 Lab
## Build Your First Agentic Pipeline

**Course:** Agentic AI: A Professional 30-Hour Course  
**Module:** 1 of 9 — Agentic AI Foundations  
**Lab hour:** Hour 3  

---

### What you will build

By the end of this notebook you will have a working **3-component agentic pipeline** running in your browser:

```
User Goal
   ↓
[PLANNER]   generate_plan()        ← LLM Call 1
   ↓
[STATE]     extract_nth_step()     ← Pure Python (no LLM)
   ↓
[EXECUTOR]  execute_step()         ← LLM Call 2
   ↓
[EVALUATOR] evaluate_output()      ← LLM Call 3
   ↓
Final Output + Token Report
```

### Colour convention (same as the board)
| Colour | Component | Role |
|--------|-----------|------|
| 🔵 Blue | Model | The LLM |
| 🟡 Amber | Tools | External functions |
| 🟢 Green | Memory | State & context |
| 🟣 Purple | Planner | Goal → steps |
| 🩵 Teal | Executor | Step → output |
| 🔴 Red | Evaluator | Quality judge |

---

> **Run cells in order from top to bottom.** Each section builds on the previous one.
> If you restart the runtime, re-run all cells from Cell 1.

---
## 🔑 SECTION 0 — API Key Setup
### How to get your API key (step-by-step)

This notebook supports **two providers**. Choose one — you only need one key.

---

#### Option A — OpenAI (recommended for beginners)

**Models available:** `gpt-4o-mini` (cheap, fast) · `gpt-4o` (powerful)

1. Go to **https://platform.openai.com**
2. Click **Sign up** → create account (or log in)
3. Click your profile icon (top-right) → **View API keys**
4. Click **+ Create new secret key**
5. Give it a name: `agentic-ai-course`
6. Copy the key — it starts with `sk-...`  
   ⚠️ You only see it once. Paste it somewhere safe.
7. Go to **Billing → Add payment method** and add $5–10 credit  
   (This lab uses < $0.10 of API calls)

---

#### Option B — Anthropic (Claude models)

**Models available:** `claude-haiku-4-5-20251001` (fast) · `claude-sonnet-4-6` (powerful)

1. Go to **https://console.anthropic.com**
2. Click **Sign up** → create account (or log in)
3. In the left sidebar, click **API Keys**
4. Click **+ Create Key**
5. Name it: `agentic-ai-course`
6. Copy the key — it starts with `sk-ant-...`
7. Go to **Billing** → add credit (minimum $5)

---

#### Storing your key safely in Colab

**Never paste your API key directly into a code cell.** Use Colab Secrets:

1. Click the 🔑 **key icon** in the left sidebar of Colab
2. Click **+ Add new secret**
3. Name: `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`)
4. Value: paste your key
5. Toggle **Notebook access** to ON
6. Run the setup cell below

![Colab Secrets location](https://i.imgur.com/placeholder.png)

> **Why secrets?** Colab notebooks can be shared. If your key is in a code cell,
> anyone you share the notebook with can see and use it. Secrets are stored
> encrypted per-user and never visible in the notebook.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0A — Install packages                          ║
# ║  Run this first. Takes ~15 seconds.                  ║
# ╚══════════════════════════════════════════════════════╝

print("📦 Installing packages...")
import subprocess
subprocess.run(["pip", "install", "openai", "anthropic", "--quiet"], check=True)
print("✅ Done — openai and anthropic installed")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0B — Choose your provider and load API key    ║
# ║                                                      ║
# ║  ► Change PROVIDER to "openai" or "anthropic"        ║
# ║  ► Make sure your key is saved in Colab Secrets      ║
# ║    (🔑 icon in left sidebar)                         ║
# ╚══════════════════════════════════════════════════════╝

# ── CHOOSE YOUR PROVIDER ──────────────────────────────────────────
PROVIDER = "openai"          # ← change to "anthropic" if using Claude
# ─────────────────────────────────────────────────────────────────

import os

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if PROVIDER == "openai":
    MODEL = "gpt-4o-mini"
    if IN_COLAB:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # If running locally: set OPENAI_API_KEY in your shell before launching
    import openai
    client = openai.OpenAI()
    print(f"✅ Provider: OpenAI | Model: {MODEL}")

elif PROVIDER == "anthropic":
    MODEL = "claude-haiku-4-5-20251001"
    if IN_COLAB:
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    import anthropic
    client = anthropic.Anthropic()
    print(f"✅ Provider: Anthropic | Model: {MODEL}")

else:
    raise ValueError(f"Unknown provider: '{PROVIDER}'. Use 'openai' or 'anthropic'.")

print(f"\n🔑 API key loaded {'from Colab Secrets' if IN_COLAB else 'from environment'}")
print("\n📋 Ready — proceed to Cell 1")

---
## 📊 SECTION 0C — Token Tracking & Cost Estimation

### What are tokens?

LLMs don't read words — they read **tokens**. A token is roughly:
- ~4 characters of English text
- ~¾ of a word on average
- `"Hello, world!"` ≈ 4 tokens
- A 500-word essay ≈ 650–700 tokens

**Why it matters:**
- API pricing is per token (input + output separately)
- Context windows have token limits
- Agentic systems make *multiple* LLM calls — tokens add up fast

### Approximate pricing (April 2026)

| Model | Input | Output |
|-------|-------|--------|
| `gpt-4o-mini` | $0.15 / 1M tokens | $0.60 / 1M tokens |
| `gpt-4o` | $2.50 / 1M tokens | $10.00 / 1M tokens |
| `claude-haiku-4-5-20251001` | $0.80 / 1M tokens | $4.00 / 1M tokens |
| `claude-sonnet-4-6` | $3.00 / 1M tokens | $15.00 / 1M tokens |

The tracker below records every LLM call in this notebook and shows a running cost total.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0C — Token Tracker (runs silently in bg)      ║
# ║  Every LLM call in this notebook is recorded here.  ║
# ╚══════════════════════════════════════════════════════╝

# Pricing per 1M tokens (update if providers change rates)
PRICING = {
    "gpt-4o-mini":              {"input": 0.15,  "output": 0.60},
    "gpt-4o":                   {"input": 2.50,  "output": 10.00},
    "claude-haiku-4-5-20251001":{"input": 0.80,  "output": 4.00},
    "claude-sonnet-4-6":        {"input": 3.00,  "output": 15.00},
}

class TokenTracker:
    """Records token usage and computes estimated cost for every LLM call."""

    def __init__(self):
        self.calls = []
        self.total_input  = 0
        self.total_output = 0
        self.total_cost   = 0.0

    def record(self, component: str, model: str,
               input_tokens: int, output_tokens: int):
        """Call this after every LLM response."""
        pricing = PRICING.get(model, {"input": 0, "output": 0})
        cost = (
            input_tokens  * pricing["input"]  / 1_000_000 +
            output_tokens * pricing["output"] / 1_000_000
        )
        self.calls.append({
            "component":     component,
            "model":         model,
            "input_tokens":  input_tokens,
            "output_tokens": output_tokens,
            "cost_usd":      cost,
        })
        self.total_input  += input_tokens
        self.total_output += output_tokens
        self.total_cost   += cost

    def report(self):
        """Print a formatted usage table."""
        sep = "─" * 72
        print(f"\n{'═'*72}")
        print(f"  TOKEN USAGE REPORT")
        print(f"{'═'*72}")
        print(f"  {'Call #':<6} {'Component':<14} {'Model':<28} {'In':>6} {'Out':>6} {'Cost':>10}")
        print(f"  {sep}")
        for i, c in enumerate(self.calls, 1):
            print(
                f"  {i:<6} {c['component']:<14} {c['model']:<28} "
                f"{c['input_tokens']:>6,} {c['output_tokens']:>6,} "
                f"${c['cost_usd']:>8.5f}"
            )
        print(f"  {sep}")
        print(
            f"  {'TOTAL':<6} {'':<14} {'':<28} "
            f"{self.total_input:>6,} {self.total_output:>6,} "
            f"${self.total_cost:>8.5f}"
        )
        print(f"{'═'*72}")
        print(f"  💡 Total tokens used: {self.total_input + self.total_output:,}")
        print(f"  💰 Estimated cost:    ${self.total_cost:.5f} USD")
        tokens_left = 1_000_000 - (self.total_input + self.total_output)
        print(f"  📦 1M token budget remaining: ~{tokens_left:,} tokens")
        print(f"{'═'*72}\n")

    def reset(self):
        self.__init__()

# Create the global tracker — used by all cells below
tracker = TokenTracker()

print("✅ Token tracker ready — it records every LLM call automatically")
print(f"   Pricing loaded for: {', '.join(PRICING.keys())}")

---
## 🔌 SECTION 0D — Unified LLM Caller

Both OpenAI and Anthropic have slightly different API shapes.  
This helper normalises them so the rest of the notebook works identically  
regardless of which provider you chose in Cell 0B.

**You do not need to modify this cell.** Just run it.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0D — Unified LLM call wrapper                 ║
# ║  Abstracts OpenAI vs Anthropic differences.         ║
# ╚══════════════════════════════════════════════════════╝

def llm_call(
    system_prompt: str,
    user_message: str,
    component_label: str = "UNKNOWN",
    max_tokens: int = 800,
) -> str:
    """
    Unified LLM caller that works with both OpenAI and Anthropic.
    Automatically records token usage in the global tracker.

    Args:
        system_prompt:   The system-level instruction (sets the agent role).
        user_message:    The user-level input (goal, step, output to evaluate, etc.).
        component_label: Label for the token report (PLANNER, EXECUTOR, etc.).
        max_tokens:      Maximum output tokens to generate.

    Returns:
        The model's response as a plain string.
    """
    global client, MODEL, PROVIDER, tracker

    if PROVIDER == "openai":
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[
                {"role": "system",  "content": system_prompt},
                {"role": "user",    "content": user_message},
            ],
        )
        text          = response.choices[0].message.content
        input_tokens  = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

    elif PROVIDER == "anthropic":
        response = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_message},
            ],
        )
        text          = response.content[0].text
        input_tokens  = response.usage.input_tokens
        output_tokens = response.usage.output_tokens

    # Record in tracker
    tracker.record(component_label, MODEL, input_tokens, output_tokens)

    return text


# ── Quick test call ───────────────────────────────────────────────
print("🧪 Running a quick test call to verify your API key...")
try:
    test_response = llm_call(
        system_prompt="You are a helpful assistant. Reply in exactly 10 words.",
        user_message="What is an AI agent in one sentence?",
        component_label="TEST",
        max_tokens=30,
    )
    print(f"\n✅ API key works!")
    print(f"   Test response: {test_response}")
    print(f"\n📊 Token usage so far:")
    tracker.report()
    tracker.reset()   # Reset after test — don't count the test call
    print("🔄 Tracker reset. Ready for the lab.")
except Exception as e:
    print(f"\n❌ API call failed: {e}")
    print("   Check your API key in Colab Secrets and try again.")

---
## 🟣 SECTION 1 — PLANNER Component

The **Planner** receives a high-level goal and decomposes it into a numbered list of concrete steps.

**Architecture position:**
```
Goal → [PLANNER] → plan (numbered list)
```

**Key teaching point:** The system prompt defines the *role* of the model for this call.  
Same model, different system prompt = different agent behaviour.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — PLANNER Component                         ║
# ║  Component: 🟣 Planner                               ║
# ║  LLM calls: 1                                        ║
# ╚══════════════════════════════════════════════════════╝

PLANNER_SYSTEM_PROMPT = """
You are a planning agent. Given a goal, return a numbered list of 4-5 concrete,
specific steps to achieve it. Each step must be:
  - Actionable (starts with a verb: Search, Write, Identify, Draft...)
  - Specific (says exactly what to do, not "do research")
  - Unambiguous (one clear action per step)

Return ONLY the numbered list. No introduction, no conclusion, no commentary.
Format exactly like this:
1. [verb] [specific action]
2. [verb] [specific action]
...
""".strip()


def generate_plan(goal: str) -> str:
    """
    [PLANNER COMPONENT]
    Receives a high-level goal and returns a numbered plan.

    Args:
        goal: The objective to plan for.
    Returns:
        A string with 4-5 numbered steps.
    """
    return llm_call(
        system_prompt=PLANNER_SYSTEM_PROMPT,
        user_message=goal,
        component_label="PLANNER",
        max_tokens=300,
    )


# ── Test the planner ──────────────────────────────────────────────
TEST_GOAL = (
    "Research and write a 3-paragraph summary of quantum computing "
    "for a software developer audience new to the topic."
)

print("🟣 [PLANNER] Sending goal to the model...")
print(f"   Goal: {TEST_GOAL}\n")

plan = generate_plan(TEST_GOAL)

print("📋 PLAN GENERATED:")
print("─" * 60)
print(plan)
print("─" * 60)

# Show token usage after this call
print("\n📊 Token usage after Planner call:")
tracker.report()

---
## 🟩 SECTION 2 — State Management (No LLM)

**State management is just Python.** Not every step in an agentic pipeline is an LLM call.
Here we parse the plan string to extract individual steps — pure string processing.

**Key teaching point:** The `plan` variable is *state* — data that passes between pipeline components.  
Notice these functions cost **zero tokens and zero dollars.**

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — State Management (no LLM call)            ║
# ║  Component: ⚙️ State / Memory                        ║
# ║  LLM calls: 0   Tokens used: 0                      ║
# ╚══════════════════════════════════════════════════════╝

import re

def extract_nth_step(plan: str, n: int = 1) -> str:
    """
    [STATE MANAGEMENT — not an LLM call]
    Parses the plan and returns step n (1-indexed).

    Handles:
      - Numbered lists: "1. Step text"
      - Dash lists:     "- Step text"
      - Plain lines:    "Step text"

    Args:
        plan: The full plan string from generate_plan().
        n:    Step number to extract (1 = first step).
    Returns:
        The text of step n, cleaned of numbering.
    """
    lines = [line.strip() for line in plan.strip().split('\n') if line.strip()]
    # Strip leading numbering or bullets ("1.", "1)", "-", "•")
    cleaned = [re.sub(r'^[\d]+[\.\)\:]\s*|^[-•*]\s*', '', l) for l in lines]
    if not cleaned:
        return plan
    idx = min(n - 1, len(cleaned) - 1)   # clamp to available steps
    return cleaned[idx]


def count_steps(plan: str) -> int:
    """Returns the number of steps in a plan string."""
    lines = [l.strip() for l in plan.strip().split('\n') if l.strip()]
    return len(lines)


def get_all_steps(plan: str) -> list:
    """Returns all steps as a clean Python list."""
    lines = [l.strip() for l in plan.strip().split('\n') if l.strip()]
    return [re.sub(r'^[\d]+[\.\)\:]\s*|^[-•*]\s*', '', l) for l in lines]


# ── Test the state parsers ────────────────────────────────────────
step_count = count_steps(plan)
all_steps  = get_all_steps(plan)

print(f"⚙️  [STATE] Plan contains {step_count} steps\n")

for i, step in enumerate(all_steps, 1):
    print(f"   Step {i}: {step}")

print()
step_1 = extract_nth_step(plan, 1)
step_2 = extract_nth_step(plan, 2)
print(f"✅ Extracted step 1: {step_1}")
print(f"✅ Extracted step 2: {step_2}")
print("\n💡 Notice: no API calls, no tokens used, no cost — just Python.")

---
## 🩵 SECTION 3 — EXECUTOR Component

The **Executor** takes a specific step and carries it out, producing a concrete output.

**Architecture position:**
```
[PLANNER] → plan → [STATE] → step → [EXECUTOR] → output
```

**Key teaching point:** The executor uses a *different system prompt* than the planner.  
Same model — different role — different behaviour.  
The `context` parameter allows step 2 to know what step 1 produced.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — EXECUTOR Component                        ║
# ║  Component: 🩵 Executor                              ║
# ║  LLM calls: 1 per step                              ║
# ╚══════════════════════════════════════════════════════╝

EXECUTOR_SYSTEM_PROMPT = """
You are an executor agent. You receive a specific task and carry it out.
Your job is to ACT, not to plan.

Rules:
  - Produce concrete, detailed output for the given task
  - Do NOT generate a new plan or list of steps
  - If context from previous steps is provided, use it to inform your work
  - Be thorough but focused on exactly what the task asks for
""".strip()


def execute_step(step: str, context: str = "", step_num: int = 1) -> str:
    """
    [EXECUTOR COMPONENT]
    Carries out a specific step, optionally using prior context.

    Args:
        step:     The specific task to perform.
        context:  Output from previous steps (passed as background info).
        step_num: Which step number this is (for tracker labelling).
    Returns:
        The concrete output of performing the step.
    """
    if context:
        user_msg = (
            f"CONTEXT FROM PREVIOUS STEPS:\n{context}\n"
            f"{'─'*40}\n"
            f"CURRENT TASK:\n{step}"
        )
    else:
        user_msg = step

    return llm_call(
        system_prompt=EXECUTOR_SYSTEM_PROMPT,
        user_message=user_msg,
        component_label=f"EXECUTOR-S{step_num}",
        max_tokens=600,
    )


# ── Execute Step 1 ────────────────────────────────────────────────
print(f"🩵 [EXECUTOR] Executing Step 1...")
print(f"   Task: {step_1}\n")

output_1 = execute_step(step_1, step_num=1)

print("📄 STEP 1 OUTPUT:")
print("─" * 60)
print(output_1)
print("─" * 60)

print("\n📊 Token usage after Executor Step 1:")
tracker.report()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3B — Execute Step 2 (with context handoff)   ║
# ║                                                      ║
# ║  Notice: output_1 is passed as context to step 2.   ║
# ║  This is the simplest form of inter-step memory.    ║
# ╚══════════════════════════════════════════════════════╝

print(f"🩵 [EXECUTOR] Executing Step 2 (with Step 1 context)...")
print(f"   Task: {step_2}\n")
print("   Context passed: first 80 chars of Step 1 output:")
print(f"   '{output_1[:80]}...'\n")

output_2 = execute_step(step_2, context=output_1, step_num=2)

print("📄 STEP 2 OUTPUT:")
print("─" * 60)
print(output_2)
print("─" * 60)

print("\n📊 Token usage after Executor Step 2:")
tracker.report()

---
## 🔴 SECTION 4 — EVALUATOR Component

The **Evaluator** reviews the output of the Executor against the original goal  
and produces a quality score + one specific improvement suggestion.

**Architecture position:**
```
[EXECUTOR] → output → [EVALUATOR] → score + critique → [EXECUTOR] (revision)
```

**Key teaching point:** The evaluator creates the *feedback loop*.  
Without an evaluator, the system runs once and stops.  
With an evaluator, the system can iterate toward quality.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — EVALUATOR Component                       ║
# ║  Component: 🔴 Evaluator                             ║
# ║  LLM calls: 1                                        ║
# ╚══════════════════════════════════════════════════════╝

EVALUATOR_SYSTEM_PROMPT = """
You are a quality evaluator for an AI pipeline.
Your job is to assess whether the output meets the original goal.

Respond in EXACTLY this format (no other text):

SCORE: [1-10]
PASS: [YES if score >= 7, NO otherwise]
STRENGTH: [One sentence — what the output does well]
IMPROVEMENT: [One specific, actionable change that would raise the score]
REVISED_PROMPT: [A revised user message to send back to the executor to implement the improvement]
""".strip()


def evaluate_output(output: str, goal: str) -> dict:
    """
    [EVALUATOR COMPONENT]
    Reviews output against the original goal.
    Returns a structured evaluation dict.

    Args:
        output: The text to evaluate.
        goal:   The original goal the output should serve.
    Returns:
        dict with keys: score, passed, strength, improvement, revised_prompt, raw
    """
    user_msg = (
        f"ORIGINAL GOAL:\n{goal}\n"
        f"{'─'*40}\n"
        f"OUTPUT TO EVALUATE:\n{output}"
    )

    raw = llm_call(
        system_prompt=EVALUATOR_SYSTEM_PROMPT,
        user_message=user_msg,
        component_label="EVALUATOR",
        max_tokens=300,
    )

    # Parse the structured response
    result = {"raw": raw, "score": 0, "passed": False,
              "strength": "", "improvement": "", "revised_prompt": ""}
    for line in raw.strip().split('\n'):
        if line.startswith("SCORE:"):
            try:
                result["score"] = int(line.split(":",1)[1].strip())
            except ValueError:
                pass
        elif line.startswith("PASS:"):
            result["passed"] = "YES" in line.upper()
        elif line.startswith("STRENGTH:"):
            result["strength"] = line.split(":",1)[1].strip()
        elif line.startswith("IMPROVEMENT:"):
            result["improvement"] = line.split(":",1)[1].strip()
        elif line.startswith("REVISED_PROMPT:"):
            result["revised_prompt"] = line.split(":",1)[1].strip()
    return result


# ── Evaluate Step 2's output ──────────────────────────────────────
print("🔴 [EVALUATOR] Reviewing Step 2 output against original goal...\n")

eval_result = evaluate_output(output_2, TEST_GOAL)

score  = eval_result['score']
passed = eval_result['passed']
icon   = "✅" if passed else "❌"

print(f"{icon} EVALUATION RESULT")
print("─" * 60)
print(f"  Score:       {score}/10")
print(f"  Pass:        {'YES — meets quality threshold' if passed else 'NO — revision needed'}")
print(f"  Strength:    {eval_result['strength']}")
print(f"  Improvement: {eval_result['improvement']}")
print(f"  Revised prompt for executor:")
print(f"    → {eval_result['revised_prompt']}")
print("─" * 60)

print("\n📊 Token usage after Evaluator:")
tracker.report()

---
## 🔁 SECTION 5 — The Full Pipeline with Revision Loop

Now we wire all four components into one function that:
1. Plans
2. Executes all steps
3. Evaluates the final output
4. Revises if score < 7 (up to `max_revisions` times)
5. Reports full token usage

**This is the complete agentic loop in ~60 lines of Python.**

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 5 — Full Agentic Pipeline with Revision Loop  ║
# ║                                                      ║
# ║  Components: Planner → Executor(s) → Evaluator      ║
# ║  Features:   revision loop, token tracking,         ║
# ║              context handoff between steps           ║
# ╚══════════════════════════════════════════════════════╝

def run_agentic_pipeline(
    goal: str,
    max_steps: int = 3,
    max_revisions: int = 2,
    quality_threshold: int = 7,
    verbose: bool = True,
) -> dict:
    """
    Runs the full agentic pipeline:
      [PLANNER] → [EXECUTOR x N] → [EVALUATOR] → (revision loop) → Output

    Args:
        goal:               The high-level objective.
        max_steps:          Maximum number of plan steps to execute.
        max_revisions:      Maximum evaluator-driven revisions.
        quality_threshold:  Score (1-10) that constitutes a passing output.
        verbose:            Print progress during execution.

    Returns:
        dict with: plan, steps_executed, final_output, eval_result, revisions_done
    """

    divider = "═" * 62
    thin    = "─" * 62

    def log(msg):
        if verbose:
            print(msg)

    # ── Reset tracker for this run ────────────────────────────────
    tracker.reset()

    log(divider)
    log(f"  🤖 AGENTIC PIPELINE")
    log(f"  Goal: {goal[:80]}{'...' if len(goal)>80 else ''}")
    log(divider)

    # ── STEP 1: PLANNER ───────────────────────────────────────────
    log("\n🟣 [PLANNER] Generating plan...")
    plan = generate_plan(goal)
    all_steps = get_all_steps(plan)
    steps_to_run = all_steps[:max_steps]
    log(f"   Plan has {len(all_steps)} steps — executing first {len(steps_to_run)}")
    for i, s in enumerate(steps_to_run, 1):
        log(f"   {i}. {s}")

    # ── STEP 2: EXECUTOR (all steps, with chained context) ────────
    log("\n🩵 [EXECUTOR] Running steps...")
    context = ""
    step_outputs = []

    for i, step in enumerate(steps_to_run, 1):
        log(f"\n   Step {i}/{len(steps_to_run)}: {step[:60]}")
        output = execute_step(step, context=context, step_num=i)
        step_outputs.append(output)
        # Chain: pass this step's output as context to the next step
        context = output
        log(f"   Output ({len(output)} chars): {output[:100]}..." if len(output) > 100 else f"   Output: {output}")

    final_output = step_outputs[-1]

    # ── STEP 3: EVALUATOR + REVISION LOOP ────────────────────────
    log(f"\n🔴 [EVALUATOR] Evaluating final output (threshold: {quality_threshold}/10)...")
    revisions_done = 0
    eval_result = None

    for revision in range(max_revisions + 1):    # +1 for the initial evaluation
        eval_result = evaluate_output(final_output, goal)
        score  = eval_result['score']
        passed = eval_result['passed'] or score >= quality_threshold

        icon = "✅" if passed else "🔁"
        log(f"   {icon} Score: {score}/10 | Pass: {'YES' if passed else 'NO'}")
        log(f"      Improvement: {eval_result['improvement']}")

        if passed or revision == max_revisions:
            if not passed:
                log(f"   ⚠️  Max revisions ({max_revisions}) reached — delivering best output")
            break

        # Revision: send the improved prompt back to the executor
        revised_task = eval_result.get('revised_prompt', '') or eval_result['improvement']
        log(f"\n   🔁 [REVISION {revision+1}] Sending revised task to executor...")
        final_output = execute_step(
            step=revised_task,
            context=final_output,
            step_num=f"R{revision+1}",
        )
        revisions_done += 1

    # ── FINAL OUTPUT ──────────────────────────────────────────────
    log(f"\n{divider}")
    log(f"  ✅ PIPELINE COMPLETE")
    log(f"  Revisions performed: {revisions_done}")
    log(f"  Final score: {eval_result['score']}/10")
    log(f"{divider}")
    log(f"\n📝 FINAL OUTPUT:")
    log(thin)
    log(final_output)
    log(thin)

    # ── TOKEN REPORT ──────────────────────────────────────────────
    tracker.report()

    return {
        "plan":            plan,
        "steps_executed":  steps_to_run,
        "step_outputs":    step_outputs,
        "final_output":    final_output,
        "eval_result":     eval_result,
        "revisions_done":  revisions_done,
    }


# ── Run the full pipeline ─────────────────────────────────────────
print("🚀 Running the complete agentic pipeline...\n")

result = run_agentic_pipeline(
    goal=TEST_GOAL,
    max_steps=3,
    max_revisions=1,
    quality_threshold=7,
    verbose=True,
)

---
## 🧪 SECTION 6 — Exercises

Complete these exercises in order. Each one extends the pipeline you just built.

---

### Exercise 1 — Change the goal (5 min)

Run the pipeline with a different goal. Observe how the planner adapts.  
Try goals from different domains:
- A technical writing task
- A data analysis task  
- A business communication task

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  EXERCISE 1 — Change the goal                       ║
# ║  Modify MY_GOAL below and run this cell.            ║
# ╚══════════════════════════════════════════════════════╝

# ── Change this goal ─────────────────────────────────────────────
MY_GOAL = """
Explain the difference between REST and GraphQL APIs
to a junior developer who has built REST APIs before but never used GraphQL.
Include one practical example showing where GraphQL has a clear advantage.
""".strip()
# ─────────────────────────────────────────────────────────────────

result_ex1 = run_agentic_pipeline(
    goal=MY_GOAL,
    max_steps=2,        # Keep it short for the exercise
    max_revisions=1,
    quality_threshold=7,
)

---
### Exercise 2 — Modify the Planner prompt (10 min)

The system prompt is what makes the model behave as a planner.  
Experiment: what happens when you change the planner's instructions?

Try:
- Change `4-5` steps to `2-3` steps → observe a shorter plan
- Add `"Each step must take less than 10 minutes"` → observe more focused steps
- Remove the format instructions → observe less structured output

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  EXERCISE 2 — Modify the Planner system prompt      ║
# ║  Change CUSTOM_PLANNER_PROMPT and run this cell.    ║
# ╚══════════════════════════════════════════════════════╝

# ── Modify this prompt ───────────────────────────────────────────
CUSTOM_PLANNER_PROMPT = """
You are a planning agent for a time-boxed task.
Given a goal, return a numbered list of exactly 2-3 steps.
Each step must:
  - Be completable in under 10 minutes
  - Start with an action verb
  - Be self-contained (no step depends on information from a future step)

Return ONLY the numbered list. No introduction. No commentary.
""".strip()
# ─────────────────────────────────────────────────────────────────

def generate_plan_custom(goal: str) -> str:
    """Planner using a custom system prompt."""
    return llm_call(
        system_prompt=CUSTOM_PLANNER_PROMPT,
        user_message=goal,
        component_label="PLANNER-CUSTOM",
        max_tokens=200,
    )

tracker.reset()

goal_ex2 = "Write a clear explanation of how git branching works for a developer new to version control."

print("🟣 [PLANNER] Running with CUSTOM prompt...\n")
custom_plan = generate_plan_custom(goal_ex2)

print("📋 CUSTOM PLAN:")
print("─" * 60)
print(custom_plan)
print("─" * 60)
print(f"\n   Steps in plan: {count_steps(custom_plan)}")

print("\n📊 Token usage:")
tracker.report()

print("\n💭 Compare this plan to the default planner output.")
print("   How does changing the prompt change the structure of the plan?")

---
### Exercise 3 — Observe the token cost of an agentic system (10 min)

A key insight of agentic systems: **a single user goal triggers multiple LLM calls.**  
This cell runs the pipeline with verbose token tracking to show you exactly  
how tokens accumulate across the planner → executor → evaluator chain.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  EXERCISE 3 — Token cost analysis                   ║
# ║  Observe how tokens accumulate across calls.        ║
# ╚══════════════════════════════════════════════════════╝

tracker.reset()

analysis_goal = (
    "Explain the concept of embeddings in machine learning "
    "to a Python developer who understands lists and arrays "
    "but has never worked with neural networks."
)

# Run with all 4 plan steps and 2 revisions to maximise calls
result_ex3 = run_agentic_pipeline(
    goal=analysis_goal,
    max_steps=4,
    max_revisions=2,
    quality_threshold=8,   # Higher bar → more likely to trigger revision
    verbose=False,         # Suppress verbose output — just show token report
)

print("\n" + "═"*62)
print("  TOKEN COST ANALYSIS")
print("═"*62)
total_calls = len(tracker.calls)
print(f"  Total LLM calls made for ONE user goal: {total_calls}")
print(f"  Revisions performed:  {result_ex3['revisions_done']}")
print(f"  Final quality score:  {result_ex3['eval_result']['score']}/10")
print()

tracker.report()

print("💭 Discussion questions:")
print("  1. How does the token count grow with each additional pipeline step?")
print("  2. Which component uses the most tokens — planner, executor, or evaluator?")
print("  3. If you ran 1,000 goals per day, what would the monthly API cost be?")
est_monthly = tracker.total_cost * 1000 * 30
print(f"     → Rough estimate: ${est_monthly:.2f}/month at 1,000 goals/day")

---
### Exercise 4 — Stretch: Build a domain-specific agent (15 min)

Modify the pipeline to be a **specialised agent** for a specific domain.  
You do this entirely through system prompts — no code changes.

Choose one:
- **Code review agent** — reviews Python code and suggests improvements
- **Meeting notes agent** — converts bullet points into structured meeting notes
- **Study guide agent** — creates a study guide from a topic description

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  EXERCISE 4 (STRETCH) — Domain-specific agent       ║
# ║  Build a Code Review Agent using only prompts.      ║
# ╚══════════════════════════════════════════════════════╝

tracker.reset()

# ── Domain-specific system prompts ────────────────────────────────
CODE_PLANNER_PROMPT = """
You are a code review planning agent.
Given a code review request, return a numbered list of 3 specific review tasks.
Each task should check one specific quality dimension:
correctness, readability, performance, security, or testability.
Return ONLY the numbered list.
""".strip()

CODE_EXECUTOR_PROMPT = """
You are a senior software engineer performing a code review.
When given a review task and code, provide specific, actionable feedback.
Format your feedback as:
  FINDING: [what you found]
  SEVERITY: [Low / Medium / High]
  SUGGESTION: [specific change to make]
""".strip()

CODE_EVALUATOR_PROMPT = """
You are reviewing a code review report.
Assess whether the review is thorough and actionable.
Respond in EXACTLY this format:
SCORE: [1-10]
PASS: [YES if score >= 7, NO otherwise]
STRENGTH: [what the review does well]
IMPROVEMENT: [one way to make the review more useful]
REVISED_PROMPT: [a prompt to send back to improve the review]
""".strip()

# ── Sample code to review ─────────────────────────────────────────
CODE_TO_REVIEW = """
def get_user(user_id):
    db = connect_to_db()
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    user = result[0]
    return user
""".strip()

REVIEW_GOAL = f"Review this Python function for quality issues:\n\n{CODE_TO_REVIEW}"

# ── Override the global prompts temporarily ───────────────────────
original_planner_prompt  = PLANNER_SYSTEM_PROMPT
original_executor_prompt = EXECUTOR_SYSTEM_PROMPT
original_evaluator_prompt = EVALUATOR_SYSTEM_PROMPT

# Monkey-patch for this exercise only
import builtins

def generate_plan_code(goal):
    return llm_call(CODE_PLANNER_PROMPT, goal, "CODE-PLANNER", 200)

def execute_step_code(step, context="", step_num=1):
    msg = f"CODE TO REVIEW:\n{CODE_TO_REVIEW}\n\nREVIEW TASK:\n{step}"
    if context:
        msg = f"PREVIOUS FINDINGS:\n{context}\n\n{msg}"
    return llm_call(CODE_EXECUTOR_PROMPT, msg, f"CODE-EXEC-S{step_num}", 400)

def evaluate_code_review(output, goal):
    msg = f"GOAL:\n{goal}\n\nCODE REVIEW:\n{output}"
    raw = llm_call(CODE_EVALUATOR_PROMPT, msg, "CODE-EVAL", 250)
    result = {"raw": raw, "score": 7, "passed": True,
              "strength": "", "improvement": "", "revised_prompt": ""}
    for line in raw.strip().split('\n'):
        if line.startswith("SCORE:"):
            try: result["score"] = int(line.split(":",1)[1].strip())
            except: pass
        elif line.startswith("PASS:"): result["passed"] = "YES" in line.upper()
        elif line.startswith("STRENGTH:"): result["strength"] = line.split(":",1)[1].strip()
        elif line.startswith("IMPROVEMENT:"): result["improvement"] = line.split(":",1)[1].strip()
        elif line.startswith("REVISED_PROMPT:"): result["revised_prompt"] = line.split(":",1)[1].strip()
    return result

# ── Run the code review pipeline ─────────────────────────────────
print("🔬 CODE REVIEW AGENT\n")
print(f"Code to review:\n{CODE_TO_REVIEW}\n")
print("─" * 60)

code_plan = generate_plan_code(REVIEW_GOAL)
print(f"\n🟣 [CODE-PLANNER] Review tasks:\n{code_plan}\n")

review_steps = get_all_steps(code_plan)[:3]
all_findings = []
context = ""

for i, step in enumerate(review_steps, 1):
    print(f"🩵 [CODE-EXEC] Review task {i}: {step[:50]}")
    finding = execute_step_code(step, context=context, step_num=i)
    all_findings.append(finding)
    context = finding
    print(f"   {finding[:200]}...\n" if len(finding) > 200 else f"   {finding}\n")

combined_review = "\n\n".join(all_findings)
print("🔴 [CODE-EVAL] Evaluating review quality...")
code_eval = evaluate_code_review(combined_review, REVIEW_GOAL)
print(f"   Score: {code_eval['score']}/10")
print(f"   Strength: {code_eval['strength']}")
print(f"   Improvement: {code_eval['improvement']}")

print("\n📊 Token usage for code review agent:")
tracker.report()

---
## 📊 SECTION 7 — Final Token Summary & Key Takeaways

Run this cell at the end of the session to see your total token usage across all exercises.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 — Session summary                           ║
# ║  Run this at the end of the lab session.            ║
# ╚══════════════════════════════════════════════════════╝

print("═" * 62)
print("  MODULE 1 LAB — SESSION COMPLETE")
print("═" * 62)

print("""
✅ What you built:
   • generate_plan()    → PLANNER component
   • extract_nth_step() → State management (no LLM)
   • execute_step()     → EXECUTOR component
   • evaluate_output()  → EVALUATOR component
   • run_agentic_pipeline() → Full orchestrator with revision loop
   • TokenTracker       → Observability for every LLM call

✅ Key concepts you observed:
   • The same model behaves differently with different system prompts
   • State management is just Python — not every step is an LLM call
   • Agentic systems multiply token usage — one goal = many LLM calls
   • The evaluator creates the feedback loop that enables self-improvement
   • Context chaining lets each step build on the previous step's output

✅ Architecture pattern you implemented:
   Goal
    ↓  [PLANNER]   → numbered task list
    ↓  [EXECUTOR]  → step 1 output → step 2 (with context) → ...
    ↓  [EVALUATOR] → score + critique
    ↓  (if score < threshold) → [EXECUTOR] revision
    ↓  Final output
""")

print("─" * 62)
print("📋 HOMEWORK for Module 2:")
print("""
  1. Annotate every line of your pipeline with its component label
     (PLANNER / EXECUTOR / EVALUATOR / STATE)

  2. Find one real AI product you use and identify:
     - Which of the 4 agentic properties does it exhibit?
     - Which components from today's lab would it contain?

  3. Read: https://www.anthropic.com/research/building-effective-agents
     (focus on the "Reflection" and "Tool use" sections)
""")
print("─" * 62)
print("🚀 Next: Module 2 — Prompt Fundamentals")
print("   4 hours mastering the language that agents speak.")
print("═" * 62)